# **Imports**

In [ ]:

from earthscape.constants import *
from earthscape.utils import *
from earthscape.splits.trainvaltestsplits_utils import *

import os
import glob
import shutil
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

# **Select Smoke Set**

In [4]:
##################################################
# Select Smoke Splits from Train/Val/Test Splits
##################################################

# set parameters...
str_version = VERSION.replace('.', '_')

if not os.path.isdir(SMOKE_DIR):
    os.makedirs(SMOKE_DIR)

if not os.path.isdir(SMOKE_SPLITS_DIR):
    os.makedirs(SMOKE_SPLITS_DIR)

seed = 111
set_seed(111)



##### select and save smoke splits
train = make_smoke_set(f"{DATASET_DIR}/{GLOBAL_AREAS_BASE}", f"../models/v_{str_version}_splits/v_{str_version}_train.geojson", split_size=7, threshold=0)
train.drop(columns=train.columns[2:], inplace=True)
train.to_file(f"{SMOKE_SPLITS_DIR}/smoke_v_{str_version}_train.geojson", driver='GeoJSON', index=False)

# val = make_smoke_set(GLOBAL_AREAS_BASE, f"../models/v_{str_version}_splits/v_{str_version}_val.geojson", split_size=7, threshold=0)
# val.drop(columns=val.columns[2:], inplace=True)
# val.to_file(f"{SMOKE_SPLITS_DIR}/smoke_v_{str_version}_val.geojson", driver='GeoJSON', index=False)

# test = make_smoke_set(GLOBAL_AREAS_BASE, f"../models/v_{str_version}_splits/v_{str_version}_test.geojson", split_size=7, threshold=0)
# test.drop(columns=test.columns[2:], inplace=True)
# test.to_file(f"{SMOKE_SPLITS_DIR}/smoke_v_{str_version}_test.geojson", driver='GeoJSON', index=False)

# cross_test = make_smoke_set(GLOBAL_AREAS_BASE, f"../models/v_{str_version}_splits/v_{str_version}_cross.geojson", split_size=7, threshold=0)
# cross_test.drop(columns=cross_test.columns[2:], inplace=True)
# cross_test.to_file(f"{SMOKE_SPLITS_DIR}/smoke_v_{str_version}_cross.geojson", driver='GeoJSON', index=False)


In [9]:
train.drop(columns=train.columns[2:])


,patch_id,geometry
6610,256_50_18829,"POLYGON ((4767693.242 3553558.392, 4767693.242..."
3425,256_50_9732,"POLYGON ((4711373.242 3537558.392, 4711373.242..."
268,256_50_524,"POLYGON ((4669133.242 3566358.392, 4669133.242..."
359,256_50_769,"POLYGON ((4670413.242 3542678.392, 4670413.242..."
1404,256_50_4024,"POLYGON ((4685133.242 3558038.392, 4685133.242..."
1126,256_50_3146,"POLYGON ((4681293.242 3533718.392, 4681293.242..."
101,256_50_185,"POLYGON ((4667853.242 3529878.392, 4667853.242..."


# **Save Smoke Set Files**

In [ ]:
##################################################
# Copy and save smoke set images to data folder.
##################################################

# paths to patches directories
dataset_dir = r'../data'
area_dirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]

# output directory
output_dir = r'../data/smoke/patches_smoke'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# list of all smoke patch IDs
all_patch_ids = train['patch_id'].astype(str).to_list()\
                + val['patch_id'].to_list()\
                + test['patch_id'].astype(str).to_list()\
                + cross_test['patch_id'].astype(str).to_list()


##### get paths to data splits...
paths = []
for pid in all_patch_ids:
    for ad in area_dirs:
        candidate_dir = glob.glob(f"{dataset_dir}/{ad}/patches_*")[0]
        if os.path.isdir(candidate_dir):
            match_img = glob.glob(f"{candidate_dir}/{pid}_*.tif")
            match_csv = glob.glob(f"{candidate_dir}/{pid}_*.csv")
            if len(match_img) > 0:
                paths.extend(match_img)
                paths.extend(match_csv)
            

##### save data to smoke dataset directory...
for src in paths:
    basename = os.path.basename(src)
    dst = f"{output_dir}/{basename}"
    shutil.copy2(src, dst)
    

# **Smoke Set Local Files**

## *Areas, Labels, & Patch Locations*

In [ ]:
#####################################################################################
# Save global files to mimic same structure as other local area dataset directories.
#####################################################################################

# global dataset file paths...
area_path = r'../data/earthscape_areas.csv'
label_path = r'../data/earthscape_labels.csv'
patches_path = r'../data/earthscape_patches.geojson'

# smoke set paths...
smoke_train_path = r'../models/splits/smoke_train.geojson'
smoke_val_path = r'../models/splits/smoke_val.geojson'
smoke_test_path = r'../models/splits/smoke_test.geojson'
smoke_cross_path = r'../models/splits/smoke_cross_test.geojson'

# output directory
output_dir = r'../data/smoke'


##### get smoke set patch IDs...
smoke_ids = []
for path in [smoke_train_path, smoke_val_path, smoke_test_path, smoke_cross_path]:
    gdf = gpd.read_file(path)
    ids = gdf['patch_id'].to_list()
    smoke_ids.extend(ids)


##### extract areas, labels, and patches files for smoke set...
area = pd.read_csv(area_path)
area = area.loc[area['patch_id'].isin(smoke_ids)]
area.to_csv(f"{output_dir}/smoke_256_50_areas.csv", index=False)

label = pd.read_csv(label_path)
label = label.loc[label['patch_id'].isin(smoke_ids)]
label.to_csv(f"{output_dir}/smoke_256_50_labels.csv", index=False)

patches = gpd.read_file(patches_path)
patches = patches.loc[patches['patch_id'].isin(smoke_ids)]
patches.to_file(f"{output_dir}/smoke_256_50_patches.geojson", driver='GeoJSON', index=False)

## *Image Stats*

In [ ]:
#############################################################
# Get Mean & Standard Deviations of Images for Normalization
#############################################################

# data directory
data_dir = r'../data/smoke'

# paths to images
image_paths = glob.glob(f"{data_dir}/patches_smoke/*.tif")
image_paths.sort(key=lambda x: x.lower())

# list of unique modality/channel basenames...
channel_names = glob.glob(f"{data_dir}/patches_smoke/*.tif")
channel_names = [os.path.basename(f) for f in channel_names]
channel_names = [f.split('_50_')[1] for f in channel_names]
channel_names = [f.split('_')[1:] for f in channel_names]
channel_names = ['_'.join(cn) for cn in channel_names]
channel_names = list(set(channel_names))
channel_names.sort()
channel_names[:40]

##### calculate mean and standard deviation for each image...
df = pd.DataFrame(columns=['channel', 'mean', 'sd'])

for idx, channel in enumerate(channel_names):
    
    image_paths = glob.glob(f"{data_dir}/patches_smoke/*{channel}")   # get paths to specific channels
    means = []
    sds = []

    for path in image_paths:
        with rasterio.open(path) as src:
            data = src.read(1, masked=True)          # read image data array
            valid_data = data[~data.mask].data       # get valid data only (masked array)
            means.append(np.mean(valid_data))        # calculate sample mean (each image is sample of population)
            sds.append(np.std(valid_data, ddof=1))   # calculate sample sd (each image is sample of population so df=1)

    # convert to numpy arrays...
    means = np.asarray(means)
    sds = np.asarray(sds)

    # overall mean using CLT
    overall_mean = np.mean(means)

    # sds**2 - within image variance | (means-overall_means)**2 - between image variance
    overall_sd = np.sqrt(np.mean(sds**2 + (means - overall_mean)**2))

    df.loc[idx, 'channel'] = channel
    df.loc[idx, 'mean'] = overall_mean
    df.loc[idx, 'sd'] = overall_sd


##### save image stats as .csv with patches
output_path = glob.glob(f"{data_dir}/*areas.csv")[0]
output_path = output_path.replace('areas', 'image_stats')
df.to_csv(output_path, index=False)
